# CS2 EXP-4 — CodeBERTa-small-v1 + LoRA (Coarse PEFT Tracking)

Rewritten to use **CodeBERTa-small-v1** (84M params, RoBERTa architecture) instead of NeoBERT-250M. Standard `transformers` + `peft` loading, LoRA targets `query`/`value` (RoBERTa self-attention naming), no runtime patches required.

Nested-CV rank search (r in {8, 16}) over the 5 frozen outer-development folds, canonical retrain on the winning rank, evaluation on the frozen outer holdout. Checkpointed per outer fold so a crashed/expired session can resume. Being ~1/3 the parameter count of NeoBERT, this should run noticeably faster and tolerate a larger batch size -- **re-run the smoke test (section 9) to get a real per-epoch timing before committing to the full run.**

## 1. Runtime settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "prashant"
REPO_ROOT = Path("/content/DiverseVul--IS-Project")
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DRIVE_ROOT = Path("/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData")
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

SPLIT_ID = "cs1_project_holdout20_innercv_v1"

# NOTE: confirm this is the correct parquet / column name for your abstracted dataset.
ABSTRACTED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"
CODE_COLUMN = "normalized_code"  # TODO: confirm this matches the abstracted parquet schema

OUTER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "outer_holdout" / "cs1_outer_project_holdout_manifest.parquet"
INNER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "inner_cv" / "cs1_project_grouped_5fold_manifest.parquet"

EXP4_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / "exp4_lora_codeberta_nested_v1"
EXP4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = "/content/drive/MyDrive/IntelligentSystemProject/hf_cache"

RANK_GRID = (8, 16)
EPOCHS = 3
TRAIN_BATCH_SIZE = 32      # CodeBERTa-small-v1 is ~1/3 the size of NeoBERT -> larger batch tolerated
GRAD_ACCUM_STEPS = 1       # effective batch = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS

RUN_SMOKE_TEST = True
RUN_EXP4_OFFICIAL = False  # flip to True only after the smoke test looks sane

print("Settings loaded.")
print("Abstracted parquet:", ABSTRACTED_PARQUET)
print("Output dir:", EXP4_OUTPUT_DIR)
print("Rank grid:", RANK_GRID, "| epochs:", EPOCHS, "| effective batch:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)

## 2. Mount Google Drive and clone/refresh repository

In [ ]:
from google.colab import drive
import subprocess
import sys


def run_command(command, cwd=None):
    print("$", " ".join(str(x) for x in command))
    subprocess.run(command, check=True, cwd=cwd)


drive.mount("/content/drive")

if not REPO_ROOT.exists():
    run_command(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)])
else:
    run_command(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\nRepository ready.")
run_command(["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"])

## 3. Verify GPU runtime

In [ ]:
import torch
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not torch.cuda.is_available():
    raise RuntimeError("EXP-4 requires a GPU runtime (Runtime > Change runtime type > GPU).")

DEVICE = torch.device("cuda")
print("CUDA device:", torch.cuda.get_device_name(0))
print("bfloat16 supported:", torch.cuda.is_bf16_supported())
print("Total VRAM: %.2f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 4. Install dependencies (peft is required for LoRA)

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "torch", "numpy", "pandas", "scikit-learn", "matplotlib",
    "pyarrow", "joblib", "peft", "tqdm",
], check=True)
print("Dependencies installed/verified.")

## 5. Patch `models.py`, `exp4/exp4_lora.py`, `exp4/exp4_nested_rank.py` with the CodeBERTa rewrite

Overwrites the three files locally in the cloned checkout. Commit these back to the repo once validated.

In [ ]:
models_path = SRC_DIR / "case_study_2" / "models.py"
exp4_lora_path = SRC_DIR / "case_study_2" / "exp4" / "exp4_lora.py"
exp4_nested_rank_path = SRC_DIR / "case_study_2" / "exp4" / "exp4_nested_rank.py"

models_path.write_text("\"\"\"\nShared CodeBERTa-small-v1 model utilities for Case Study 2.\n\nShared by:\n  - EXP-3 Frozen Linear Probe\n  - EXP-4 LoRA\n\nCodeBERTa-small-v1 is a standard RoBERTa-architecture encoder pretrained on\nCodeSearchNet source code. Unlike NeoBERT, it needs no trust_remote_code,\nno xformers/SwiGLU compatibility shims, and no custom runtime patches --\nit loads via plain transformers.AutoModel / AutoTokenizer.\n\nThis file must not contain experiment-specific training loops.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoModel, AutoTokenizer\n\n\nDEFAULT_CODE_MODEL = \"huggingface/CodeBERTa-small-v1\"\nDEFAULT_CODE_TOKENIZER = \"huggingface/CodeBERTa-small-v1\"  # ships its own code-trained BPE tokenizer\n\n\ndef configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:\n    \"\"\"\n    Configure Hugging Face cache and safer transfer behavior for Colab.\n    Should be called before model/tokenizer loading.\n    \"\"\"\n    if hf_cache_dir:\n        hf_cache_dir = str(hf_cache_dir)\n        os.environ.setdefault(\"HF_HOME\", hf_cache_dir)\n        os.environ.setdefault(\"HUGGINGFACE_HUB_CACHE\", str(Path(hf_cache_dir) / \"hub\"))\n\n    os.environ.setdefault(\"HF_HUB_DISABLE_XET\", \"1\")\n    os.environ.setdefault(\"HF_HUB_ENABLE_HF_TRANSFER\", \"0\")\n    os.environ.setdefault(\"HF_HUB_DOWNLOAD_TIMEOUT\", \"120\")\n    os.environ.setdefault(\"HF_HUB_ETAG_TIMEOUT\", \"120\")\n\n\ndef _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:\n    dtype_policy = (dtype_policy or \"auto\").lower()\n    device = str(device)\n\n    if dtype_policy == \"float16\":\n        return torch.float16 if device == \"cuda\" else torch.float32\n    if dtype_policy == \"bfloat16\":\n        return torch.bfloat16 if device == \"cuda\" and torch.cuda.is_bf16_supported() else torch.float32\n    if dtype_policy == \"float32\":\n        return torch.float32\n    if dtype_policy == \"auto\":\n        if device == \"cuda\" and torch.cuda.is_bf16_supported():\n            return torch.bfloat16\n        if device == \"cuda\":\n            return torch.float32\n        return torch.float32\n\n    raise ValueError(f\"Unknown dtype_policy: {dtype_policy}\")\n\n\ndef load_code_tokenizer(\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,\n    hf_cache_dir: Optional[str] = None,\n):\n    configure_huggingface_cache(hf_cache_dir)\n    return AutoTokenizer.from_pretrained(\n        tokenizer_name,\n        use_fast=True,\n        cache_dir=hf_cache_dir,\n    )\n\n\ndef load_code_encoder(\n    model_name: str = DEFAULT_CODE_MODEL,\n    dtype_policy: str = \"auto\",\n    device: Optional[str] = None,\n    freeze: bool = True,\n    hf_cache_dir: Optional[str] = None,\n) -> nn.Module:\n    \"\"\"\n    Load the CodeBERTa encoder. Standard AutoModel loading -- no runtime\n    patches required (unlike NeoBERT's custom architecture).\n    \"\"\"\n    device = device or (\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    configure_huggingface_cache(hf_cache_dir)\n\n    dtype = _dtype_from_policy(dtype_policy, device)\n    is_local_path = Path(str(model_name)).exists()\n\n    kwargs: Dict[str, Any] = {\n        \"cache_dir\": hf_cache_dir,\n        \"local_files_only\": bool(is_local_path),\n    }\n    if dtype is not None:\n        kwargs[\"torch_dtype\"] = dtype\n\n    model = AutoModel.from_pretrained(model_name, **kwargs)\n    model.to(device)\n\n    if freeze:\n        for param in model.parameters():\n            param.requires_grad = False\n        model.eval()\n\n    return model\n\n\ndef mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    \"\"\"\n    Mean-pool token embeddings using the attention mask.\n    Preferred default for frozen probes: raw CLS-token embeddings from a\n    frozen encoder without a pooling-specific pretraining objective are a\n    known weak sentence representation (Reimers & Gurevych, 2019).\n    \"\"\"\n    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)\n    summed = (last_hidden_state * mask).sum(dim=1)\n    denom = mask.sum(dim=1).clamp(min=1.0)\n    return summed / denom\n\n\ndef cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:\n    \"\"\"Use first token (<s> / CLS-equivalent) embedding.\"\"\"\n    return last_hidden_state[:, 0, :]\n\n\nclass CodeSequenceClassifier(nn.Module):\n    \"\"\"\n    Shared sequence classifier wrapper for LoRA-style fine-tuning (EXP-4).\n    EXP-3's linear probe normally only uses frozen encoder embeddings\n    extracted separately, but this class is reused wherever an end-to-end\n    trainable classification head is needed.\n    \"\"\"\n\n    def __init__(\n        self,\n        model_name: str = DEFAULT_CODE_MODEL,\n        num_labels: int = 1,\n        freeze_backbone: bool = False,\n        pooling: str = \"mean\",\n        dtype_policy: str = \"auto\",\n        hf_cache_dir: Optional[str] = None,\n    ) -> None:\n        super().__init__()\n        device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        self.backbone = load_code_encoder(\n            model_name=model_name,\n            dtype_policy=dtype_policy,\n            device=device,\n            freeze=freeze_backbone,\n            hf_cache_dir=hf_cache_dir,\n        )\n        hidden_size = int(self.backbone.config.hidden_size)\n        self.classification_head = nn.Linear(hidden_size, num_labels)\n        self.pooling = pooling\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:\n        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)\n        hidden = outputs.last_hidden_state\n        if self.pooling == \"cls\":\n            pooled = cls_pool_last_hidden(hidden)\n        else:\n            pooled = mean_pool_last_hidden(hidden, attention_mask)\n        logits = self.classification_head(pooled)\n        return logits.squeeze(-1)\n\n\ndef count_trainable_parameters(model: nn.Module) -> Dict[str, int]:\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    return {\n        \"trainable_parameters\": int(trainable),\n        \"total_parameters\": int(total),\n        \"trainable_percent\": float(100.0 * trainable / max(total, 1)),\n    }\n\n\ndef infer_lora_target_modules(model: nn.Module) -> List[str]:\n    \"\"\"\n    Infer LoRA target module names for a RoBERTa-family encoder (CodeBERTa).\n    RoBERTa self-attention uses separate `query` / `value` Linear layers\n    (not a fused qkv projection like NeoBERT), so this is the expected match.\n    \"\"\"\n    module_names = [name for name, _ in model.named_modules()]\n\n    candidate_sets = [\n        [\"query\", \"value\"],\n        [\"q_proj\", \"v_proj\"],\n        [\"qkv\"],\n        [\"in_proj\"],\n    ]\n\n    for candidates in candidate_sets:\n        if all(any(name.endswith(candidate) or f\".{candidate}\" in name for name in module_names) for candidate in candidates):\n            return candidates\n\n    return [\"query\", \"value\"]\n\n\ndef create_lora_sequence_classifier(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    lora_dropout: float = 0.05,\n    dtype_policy: str = \"auto\",\n    hf_cache_dir: Optional[str] = None,\n):\n    \"\"\"\n    Shared LoRA model creation helper for EXP-4. PEFT is imported lazily.\n    \"\"\"\n    try:\n        from peft import LoraConfig, get_peft_model\n    except Exception as exc:\n        raise ImportError(\"PEFT is required for EXP-4 LoRA. Install with `pip install peft`.\") from exc\n\n    base = CodeSequenceClassifier(\n        model_name=model_name,\n        freeze_backbone=False,\n        pooling=\"mean\",\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n    )\n\n    target_modules = infer_lora_target_modules(base)\n\n    config = LoraConfig(\n        r=rank,\n        lora_alpha=lora_alpha,\n        target_modules=target_modules,\n        lora_dropout=lora_dropout,\n        bias=\"none\",\n        task_type=\"FEATURE_EXTRACTION\",\n    )\n    return get_peft_model(base, config)\n\n\ndef get_exp4_lora_model(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    dtype_policy: str = \"auto\",\n    hf_cache_dir: Optional[str] = None,\n):\n    return create_lora_sequence_classifier(\n        model_name=model_name,\n        rank=rank,\n        lora_alpha=lora_alpha,\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n    )\n")
exp4_lora_path.write_text("import time\nimport torch\nimport torch.nn as nn\nfrom torch.optim import AdamW\nimport numpy as np\n\nfrom case_study_2.data_loader import create_dataloader, get_class_weights\nfrom case_study_2.models import get_exp4_lora_model, count_trainable_parameters, DEFAULT_CODE_MODEL\n\n\ndef train_lora_model(\n    train_df, val_df, tokenizer, rank, epochs=3, batch_size=32,\n    grad_accum_steps=1, device=\"cuda\", hf_cache_dir=None, verbose=True,\n    code_column=\"normalized_code\", max_length=512,\n):\n    \"\"\"\n    Trains a LoRA-adapted CodeBERTa-small-v1 sequence classifier.\n\n    CodeBERTa-small-v1 is a 6-layer, 84M-parameter RoBERTa-family encoder --\n    roughly a third the size of NeoBERT-250M -- so it tolerates a larger\n    default batch size (32, grad_accum=1) on a single T4/L4 GPU. Reduce\n    batch_size / raise grad_accum_steps if you see OOM errors.\n    \"\"\"\n    train_loader = create_dataloader(\n        train_df, tokenizer, batch_size=batch_size, max_length=max_length,\n        shuffle=True, num_workers=2, code_column=code_column,\n    )\n    val_loader = create_dataloader(\n        val_df, tokenizer, batch_size=64, max_length=max_length,\n        shuffle=False, num_workers=2, code_column=code_column,\n    )\n\n    model = get_exp4_lora_model(\n        model_name=DEFAULT_CODE_MODEL, rank=rank, lora_alpha=16, hf_cache_dir=hf_cache_dir,\n    ).to(device)\n\n    is_cuda = (device == \"cuda\") or (hasattr(device, \"type\") and device.type == \"cuda\")\n\n    if verbose:\n        stats = count_trainable_parameters(model)\n        print(\n            f\"    [lora] rank={rank} | trainable={stats['trainable_parameters']:,} \"\n            f\"({stats['trainable_percent']:.3f}%) | total={stats['total_parameters']:,}\"\n        )\n        if is_cuda:\n            print(f\"    [lora] VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB\")\n\n    pos_weight = get_class_weights(train_df).to(device)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    optimizer = AdamW(model.parameters(), lr=2e-4)\n\n    t0 = time.time()\n    for epoch in range(epochs):\n        model.train()\n        epoch_loss = 0.0\n        n_steps = 0\n        optimizer.zero_grad()\n\n        for step, batch in enumerate(train_loader):\n            input_ids = batch[\"input_ids\"].to(device, non_blocking=True)\n            attention_mask = batch[\"attention_mask\"].to(device, non_blocking=True)\n            labels = batch[\"label\"].to(device, non_blocking=True)\n\n            autocast_dtype = torch.bfloat16 if is_cuda and torch.cuda.is_bf16_supported() else torch.float16\n            with torch.amp.autocast(device_type=\"cuda\", dtype=autocast_dtype):\n                logits = model(input_ids, attention_mask)\n                loss = criterion(logits, labels) / grad_accum_steps\n\n            loss.backward()\n            epoch_loss += loss.item() * grad_accum_steps\n            n_steps += 1\n\n            if (step + 1) % grad_accum_steps == 0:\n                optimizer.step()\n                optimizer.zero_grad()\n\n        optimizer.step()\n        optimizer.zero_grad()\n\n        if verbose:\n            elapsed_min = (time.time() - t0) / 60\n            peak_vram = torch.cuda.max_memory_allocated() / 1e9 if is_cuda else 0.0\n            print(\n                f\"    [lora] epoch {epoch+1}/{epochs} | avg_loss={epoch_loss/max(n_steps,1):.4f} \"\n                f\"| elapsed={elapsed_min:.1f} min | peak_VRAM={peak_vram:.2f} GB\"\n            )\n\n    model.eval()\n    all_scores = []\n    with torch.no_grad():\n        for batch in val_loader:\n            input_ids = batch[\"input_ids\"].to(device, non_blocking=True)\n            attention_mask = batch[\"attention_mask\"].to(device, non_blocking=True)\n            autocast_dtype = torch.bfloat16 if is_cuda and torch.cuda.is_bf16_supported() else torch.float16\n            with torch.amp.autocast(device_type=\"cuda\", dtype=autocast_dtype):\n                logits = model(input_ids, attention_mask)\n                scores = torch.sigmoid(logits)\n            all_scores.extend(scores.float().cpu().numpy())\n\n    del train_loader, val_loader, criterion, optimizer\n    if is_cuda:\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n\n    return np.array(all_scores), model\n")
exp4_nested_rank_path.write_text("import os\nimport sys\nimport gc\nimport time\nimport torch\nimport joblib\nimport pandas as pd\nimport numpy as np\nfrom pathlib import Path\n\nCURRENT_DIR = Path(__file__).resolve().parent\nSRC_DIR = CURRENT_DIR.parents[1]\nif str(SRC_DIR) not in sys.path:\n    sys.path.insert(0, str(SRC_DIR))\n\nfrom case_study_2.data_loader import create_dataloader, get_class_weights\nfrom case_study_2.models import configure_huggingface_cache, load_code_tokenizer, DEFAULT_CODE_TOKENIZER\nfrom case_study_2.exp4.exp4_lora import train_lora_model\nfrom case_study_1 import split_manifest\nfrom sklearn.metrics import average_precision_score\n\n\ndef run_exp4_nested_pipeline(\n    abstracted_parquet_path, inner_manifest_path, outer_manifest_path,\n    output_dir_path, hf_cache_dir=None, rank_grid=(8, 16), epochs=3,\n    code_column=\"normalized_code\",\n):\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"[EXP4] Starting Nested LoRA Tuning Pipeline (CodeBERTa-small-v1) on Device: {device}\")\n\n    OUTPUT_DIR = Path(output_dir_path)\n    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    checkpoint_path = OUTPUT_DIR / \"exp4_nested_checkpoint.joblib\"\n\n    configure_huggingface_cache(hf_cache_dir)\n\n    full_df = pd.read_parquet(abstracted_parquet_path)\n    outer_manifest_df = pd.read_parquet(outer_manifest_path)\n\n    outer_cv_manifest = split_manifest.load_manifest(\n        inner_manifest_path,\n        config=split_manifest.SplitConfig(n_splits=5, random_state=42, shuffle=True),\n    )\n\n    dev_ids = set(outer_manifest_df.loc[outer_manifest_df[\"partition\"] == \"development\", \"source_row_id\"].tolist())\n    development_frame = full_df[full_df[\"source_row_id\"].isin(dev_ids)].copy().reset_index(drop=True)\n\n    tokenizer = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=hf_cache_dir)\n\n    oof_predictions_list = []\n    selected_ranks_audit = []\n    completed_outer_folds = set()\n\n    if checkpoint_path.exists():\n        ckpt = joblib.load(checkpoint_path)\n        oof_predictions_list = ckpt[\"oof_predictions_list\"]\n        selected_ranks_audit = ckpt[\"selected_ranks_audit\"]\n        completed_outer_folds = ckpt[\"completed_outer_folds\"]\n        print(f\"[EXP4] Resuming: {len(completed_outer_folds)}/5 outer folds already completed\")\n\n    for outer_fold in range(5):\n        if outer_fold in completed_outer_folds:\n            print(f\"[EXP4] Outer fold {outer_fold}: already completed, skipping.\")\n            continue\n\n        fold_t0 = time.time()\n        print(f\"\\n=================== OUTER FOLD {outer_fold + 1}/5 ===================\")\n\n        outer_train_manifest = outer_cv_manifest[outer_cv_manifest[\"fold\"] != outer_fold]\n        outer_val_manifest = outer_cv_manifest[outer_cv_manifest[\"fold\"] == outer_fold]\n\n        outer_train_df = development_frame[development_frame[\"source_row_id\"].isin(outer_train_manifest[\"source_row_id\"])]\n        outer_val_df = development_frame[development_frame[\"source_row_id\"].isin(outer_val_manifest[\"source_row_id\"])]\n\n        print(f\"  [Inner Loop] Starting inner tuning for Outer Fold {outer_fold}...\")\n        inner_cv_manifest = split_manifest.load_manifest(\n            outer_train_df[[\"source_row_id\", \"label\", \"project\"]],\n            config=split_manifest.SplitConfig(n_splits=3, random_state=20260707, shuffle=True),\n        )\n\n        rank_performance = {}\n\n        for rank_candidate in rank_grid:\n            print(f\"    Testing candidate Rank = {rank_candidate}\")\n            inner_fold_praucs = []\n\n            for inner_fold in range(3):\n                inner_t0 = time.time()\n                inner_train_ids = inner_cv_manifest[inner_cv_manifest[\"fold\"] != inner_fold][\"source_row_id\"]\n                inner_val_ids = inner_cv_manifest[inner_cv_manifest[\"fold\"] == inner_fold][\"source_row_id\"]\n\n                inner_train_df = outer_train_df[outer_train_df[\"source_row_id\"].isin(inner_train_ids)]\n                inner_val_df = outer_train_df[outer_train_df[\"source_row_id\"].isin(inner_val_ids)]\n\n                val_scores, tmp_model = train_lora_model(\n                    inner_train_df, inner_val_df, tokenizer, rank=rank_candidate,\n                    epochs=epochs, device=device, hf_cache_dir=hf_cache_dir,\n                    code_column=code_column,\n                )\n\n                prauc = average_precision_score(inner_val_df[\"label\"].values, val_scores)\n                inner_fold_praucs.append(prauc)\n\n                print(\n                    f\"      inner_fold {inner_fold}: PR-AUC={prauc:.4f} \"\n                    f\"| {(time.time()-inner_t0)/60:.1f} min\"\n                )\n\n                del tmp_model, inner_train_df, inner_val_df\n                gc.collect()\n                if device.type == \"cuda\":\n                    torch.cuda.empty_cache()\n\n            mean_inner_prauc = np.mean(inner_fold_praucs)\n            print(f\"    -> Rank {rank_candidate} | Inner Mean PR-AUC: {mean_inner_prauc:.4f}\")\n            rank_performance[rank_candidate] = mean_inner_prauc\n\n        optimal_rank = max(rank_performance, key=rank_performance.get)\n        print(f\"  [Inner Loop] Selected Optimal Rank = {optimal_rank} for Outer Fold {outer_fold}\")\n        selected_ranks_audit.append({\"outer_fold\": outer_fold, \"selected_rank\": optimal_rank})\n\n        print(f\"  [Outer Refit] Executing final refit on Outer Train with Rank = {optimal_rank}...\")\n        outer_val_scores, final_outer_model = train_lora_model(\n            outer_train_df, outer_val_df, tokenizer, rank=optimal_rank,\n            epochs=epochs, device=device, hf_cache_dir=hf_cache_dir,\n            code_column=code_column,\n        )\n\n        fold_oof_df = pd.DataFrame({\n            \"source_row_id\": outer_val_df[\"source_row_id\"].values,\n            \"project\": outer_val_df[\"project\"].values,\n            \"label\": outer_val_df[\"label\"].values.astype(int),\n            \"y_score\": outer_val_scores,\n            \"fold\": outer_fold,\n        })\n        oof_predictions_list.append(fold_oof_df)\n\n        if outer_fold == 4:\n            final_outer_model.save_pretrained(OUTPUT_DIR / \"final_exp4_lora_adapter\")\n            print(f\"  [Backup] Saved final fold LoRA adapter to: {OUTPUT_DIR / 'final_exp4_lora_adapter'}\")\n\n        del final_outer_model, outer_train_df, outer_val_df\n        gc.collect()\n        if device.type == \"cuda\":\n            torch.cuda.empty_cache()\n\n        completed_outer_folds.add(outer_fold)\n\n        joblib.dump({\n            \"oof_predictions_list\": oof_predictions_list,\n            \"selected_ranks_audit\": selected_ranks_audit,\n            \"completed_outer_folds\": completed_outer_folds,\n        }, checkpoint_path)\n\n        fold_min = (time.time() - fold_t0) / 60\n        print(f\"[EXP4] Outer fold {outer_fold} done in {fold_min:.1f} min | checkpoint saved\")\n\n    oof_df = pd.concat(oof_predictions_list).reset_index(drop=True)\n    oof_df.to_csv(OUTPUT_DIR / \"exp4_oof_predictions.csv\", index=False)\n\n    audit_ranks_df = pd.DataFrame(selected_ranks_audit)\n    audit_ranks_df.to_csv(OUTPUT_DIR / \"exp4_selected_rank_per_outer_fold.csv\", index=False)\n\n    best_global_rank = int(audit_ranks_df[\"selected_rank\"].mode()[0])\n    print(f\"\\n--- [EXP4] Starting Global Canonical Retraining with Winner Rank = {best_global_rank} ---\")\n\n    holdout_ids = set(outer_manifest_df.loc[outer_manifest_df[\"partition\"] == \"outer_holdout\", \"source_row_id\"].tolist())\n    holdout_frame = full_df[full_df[\"source_row_id\"].isin(holdout_ids)].copy().reset_index(drop=True)\n\n    holdout_scores, global_model = train_lora_model(\n        development_frame, holdout_frame, tokenizer, rank=best_global_rank,\n        epochs=epochs, device=device, hf_cache_dir=hf_cache_dir,\n        code_column=code_column,\n    )\n\n    global_model.save_pretrained(OUTPUT_DIR / \"final_canonical_lora_model\")\n    print(f\"  Production model saved to: {OUTPUT_DIR / 'final_canonical_lora_model'}\")\n\n    holdout_predictions_df = pd.DataFrame({\n        \"source_row_id\": holdout_frame[\"source_row_id\"].values,\n        \"project\": holdout_frame[\"project\"].values,\n        \"label\": holdout_frame[\"label\"].values.astype(int),\n        \"y_score\": holdout_scores,\n        \"fold\": 0,\n    })\n    holdout_predictions_df.to_csv(OUTPUT_DIR / \"exp4_holdout_predictions.csv\", index=False)\n    print(f\"[EXP4] LoRA pipeline finished. Predictions saved to output.\")\n\n    del global_model\n    gc.collect()\n    if device.type == \"cuda\":\n        torch.cuda.empty_cache()\n\n    if checkpoint_path.exists():\n        checkpoint_path.unlink()\n\n    return {\n        \"oof_predictions\": oof_df,\n        \"selected_ranks_audit\": audit_ranks_df,\n        \"holdout_predictions\": holdout_predictions_df,\n        \"best_global_rank\": best_global_rank,\n        \"output_dir\": OUTPUT_DIR,\n    }\n\n\nif __name__ == \"__main__\":\n    run_exp4_nested_pipeline(\n        abstracted_parquet_path=\"/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet\",\n        inner_manifest_path=\"/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet\",\n        outer_manifest_path=\"/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet\",\n        output_dir_path=\"/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/exp4_lora_codeberta_output\",\n        hf_cache_dir=\"/content/drive/MyDrive/IntelligentSystemProject/hf_cache\",\n    )\n")

print("Patched:", models_path)
print("Patched:", exp4_lora_path)
print("Patched:", exp4_nested_rank_path)


## 6. Verify files and import project modules

In [ ]:
required_repo_files = [
    SRC_DIR / "case_study_1" / "split_manifest.py",
    SRC_DIR / "case_study_2" / "data_loader.py",
    SRC_DIR / "case_study_2" / "models.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_lora.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_nested_rank.py",
]
missing = [str(p) for p in required_repo_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

import sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader, get_class_weights
from case_study_2.models import (
    configure_huggingface_cache, load_code_tokenizer, load_code_encoder,
    DEFAULT_CODE_MODEL, DEFAULT_CODE_TOKENIZER,
    CodeSequenceClassifier, infer_lora_target_modules, count_trainable_parameters,
)
from case_study_2.exp4.exp4_lora import train_lora_model
from case_study_2.exp4.exp4_nested_rank import run_exp4_nested_pipeline

print("Imported project modules successfully.")
print("Model:", DEFAULT_CODE_MODEL, "| Tokenizer:", DEFAULT_CODE_TOKENIZER)

## 7. Sanity-check LoRA target modules before spending any compute

In [ ]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

_probe_model = CodeSequenceClassifier(
    model_name=DEFAULT_CODE_MODEL, freeze_backbone=False,
    dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR,
)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred LoRA target_modules:", _target_modules)

_stats = count_trainable_parameters(_probe_model)
print("Full model trainable params (pre-LoRA, sanity check):", f"{_stats['total_parameters']:,}")

del _probe_model
torch.cuda.empty_cache()

## 8. Dataset & manifest existence check

In [ ]:
if not ABSTRACTED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing dataset: {ABSTRACTED_PARQUET}")
if not OUTER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing outer manifest: {OUTER_MANIFEST_PATH}")
if not INNER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing inner manifest: {INNER_MANIFEST_PATH}")

import pandas as pd
_preview = pd.read_parquet(ABSTRACTED_PARQUET)
print("Rows:", len(_preview))
print("Columns:", _preview.columns.tolist())
if CODE_COLUMN not in _preview.columns:
    raise ValueError(f"CODE_COLUMN='{CODE_COLUMN}' not found. Available columns: {_preview.columns.tolist()}")
del _preview

## 9. Smoke test — tiny subsample, 1 epoch, single rank

Verifies the LoRA training loop runs end-to-end without OOM and produces a sane PR-AUC, and gives a real per-epoch timing to recalibrate the full-run estimate.

In [ ]:
import time
import pandas as pd
from sklearn.metrics import average_precision_score

if RUN_SMOKE_TEST:
    configure_huggingface_cache(HF_CACHE_DIR)
    tokenizer_smoke = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

    full_df_smoke = pd.read_parquet(ABSTRACTED_PARQUET)
    sample_df = full_df_smoke.sample(n=min(2000, len(full_df_smoke)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)

    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")
    print(f"[smoke] VRAM before: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    t0 = time.time()
    smoke_scores, smoke_model = train_lora_model(
        smoke_train, smoke_val, tokenizer_smoke, rank=RANK_GRID[0], epochs=1,
        batch_size=TRAIN_BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=CODE_COLUMN,
    )
    smoke_prauc = average_precision_score(smoke_val["label"].values, smoke_scores)
    print(f"[smoke] Done in {(time.time()-t0)/60:.1f} min | PR-AUC={smoke_prauc:.4f}")
    print(f"[smoke] Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

    del smoke_model, sample_df, smoke_train, smoke_val, full_df_smoke
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
else:
    print("RUN_SMOKE_TEST=False; skipping.")

## 10. Official nested LoRA pipeline (checkpointed, resumable)

Set `RUN_EXP4_OFFICIAL = True` in section 1 once the smoke test looks sane. If a session crashes or expires mid-run, re-run this cell: it loads `exp4_nested_checkpoint.joblib` from Drive and skips already-completed outer folds.

In [ ]:
if RUN_EXP4_OFFICIAL:
    exp4_results = run_exp4_nested_pipeline(
        abstracted_parquet_path=str(ABSTRACTED_PARQUET),
        inner_manifest_path=str(INNER_MANIFEST_PATH),
        outer_manifest_path=str(OUTER_MANIFEST_PATH),
        output_dir_path=str(EXP4_OUTPUT_DIR),
        hf_cache_dir=HF_CACHE_DIR,
        rank_grid=RANK_GRID,
        epochs=EPOCHS,
        code_column=CODE_COLUMN,
    )
    print("\nOfficial EXP-4 (CodeBERTa) nested LoRA run complete.")
else:
    exp4_results = None
    print("RUN_EXP4_OFFICIAL=False; official nested run skipped.")

## 11. Display nested result summary

In [ ]:
from sklearn.metrics import average_precision_score

if exp4_results is None:
    print("No results in memory. Set RUN_EXP4_OFFICIAL=True and re-run section 10,")
    print("or reload from disk below if a previous run already completed.")
else:
    print("Selected rank by outer fold:")
    display(exp4_results["selected_ranks_audit"])

    oof = exp4_results["oof_predictions"]
    pooled_prauc = average_precision_score(oof["label"].values, oof["y_score"].values)
    print(f"\nPooled nested OOF PR-AUC: {pooled_prauc:.4f}")
    print(f"Best global rank (mode across folds): {exp4_results['best_global_rank']}")

    holdout_pred = exp4_results["holdout_predictions"]
    holdout_prauc = average_precision_score(holdout_pred["label"].values, holdout_pred["y_score"].values)
    print(f"Outer holdout PR-AUC: {holdout_prauc:.4f}")

## 11b. Reload results from disk (if the kernel restarted after the run finished)

In [ ]:
import pandas as pd
from sklearn.metrics import average_precision_score

oof_path = EXP4_OUTPUT_DIR / "exp4_oof_predictions.csv"
holdout_path = EXP4_OUTPUT_DIR / "exp4_holdout_predictions.csv"
ranks_path = EXP4_OUTPUT_DIR / "exp4_selected_rank_per_outer_fold.csv"

if oof_path.exists():
    oof_reloaded = pd.read_csv(oof_path)
    print("Pooled nested OOF PR-AUC (reloaded):",
          average_precision_score(oof_reloaded["label"].values, oof_reloaded["y_score"].values))
    display(pd.read_csv(ranks_path))
if holdout_path.exists():
    holdout_reloaded = pd.read_csv(holdout_path)
    print("Outer holdout PR-AUC (reloaded):",
          average_precision_score(holdout_reloaded["label"].values, holdout_reloaded["y_score"].values))

## 12. Compare against previous experiments

In [ ]:
from sklearn.metrics import average_precision_score
import pandas as pd

reference_rows = [
    {"experiment": "EXP-0 fixed normalized_code", "scope": "development pooled OOF", "pr_auc": 0.125205},
    {"experiment": "EXP-0 nested-alpha normalized_code", "scope": "development nested pooled OOF", "pr_auc": 0.141971},
    {"experiment": "EXP-2 MLP fixed representation", "scope": "development pooled OOF", "pr_auc": 0.138157},
    {"experiment": "EXP-2/CS2 NeoBERT frozen linear probe (CLS)", "scope": "development nested pooled OOF", "pr_auc": 0.093483},
]

if exp4_results is not None:
    oof = exp4_results["oof_predictions"]
    reference_rows.append({
        "experiment": "EXP-3/CS2 CodeBERTa + LoRA",
        "scope": "development nested pooled OOF",
        "pr_auc": float(average_precision_score(oof["label"].values, oof["y_score"].values)),
    })

display(pd.DataFrame(reference_rows).sort_values("pr_auc", ascending=False).reset_index(drop=True))

## 13. Cleanup GPU memory

In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")